# 🧪 W2-D4 概念实验：位置编码深入 — RoPE 与 ALiBi

> 配套阅读：`第2周-Day4-位置编码深入-RoPE与ALiBi.md`（四种方案对比、NTK 外推、业务关联在那边）
> 这个 notebook 用可执行实验回答三个问题：
> 1. **RoPE 的核心魔法：为什么 q·k 只依赖相对位置？** 用矩阵旋转亲手验证
> 2. **RoPE 注意力分数的热力图** — 对角线结构
> 3. **ALiBi 的"越远越不重要"机制** — 线性偏置如何塑造注意力分布

## 实验 1：RoPE 旋转 — 验证 q_m·k_n 只取决于 (m−n)

RoPE 将位置 m 的 q 向量和位置 n 的 k 向量分别旋转，使得点积只依赖相对位置差。这是 RoPE 最核心的性质。

In [ ]:
import numpy as np

np.random.seed(42)
d = 8  # 维度（必须是偶数）

# RoPE 的基频 theta_i = 10000^(-2i/d)
thetas = 10000.0 ** (-2.0 * np.arange(d // 2) / d)

def rope_rotate(x, pos):
    x_out = x.copy()
    for i in range(d // 2):
        theta = thetas[i]
        cos_t, sin_t = np.cos(pos * theta), np.sin(pos * theta)
        x_out[2*i]     = x[2*i] * cos_t - x[2*i+1] * sin_t
        x_out[2*i + 1] = x[2*i] * sin_t + x[2*i+1] * cos_t
    return x_out

q = np.random.randn(d)
k = np.random.randn(d)

print("=== 验证 RoPE 相对位置性质 ===")
print(f"{'位置 m':>6} {'位置 n':>6} {'m-n':>4} {'q_m · k_n':>12}")
print("-" * 35)

for m, n in [(2,0), (3,1), (5,3), (10,8), (100,98)]:
    q_m = rope_rotate(q, m)
    k_n = rope_rotate(k, n)
    print(f"{m:>6} {n:>6} {m-n:>4} {np.dot(q_m, k_n):>12.6f}")

print()
for m, n in [(5,0), (8,3), (10,5), (50,45)]:
    q_m = rope_rotate(q, m)
    k_n = rope_rotate(k, n)
    print(f"{m:>6} {n:>6} {m-n:>4} {np.dot(q_m, k_n):>12.6f}")

print("\n结论：相同 m-n 的点积完全一致 → RoPE 编码的是相对位置！")

## 实验 2：RoPE 注意力分数热力图 — 对角线结构

相同 (m−n) 的位置分数相同 → 热力图沿对角线呈条纹状。

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

np.random.seed(0)
seq_len = 16
d = 8
thetas = 10000.0 ** (-2.0 * np.arange(d // 2) / d)

def rope_rotate(x, pos):
    x_out = x.copy()
    for i in range(d // 2):
        cos_t, sin_t = np.cos(pos * thetas[i]), np.sin(pos * thetas[i])
        x_out[2*i]     = x[2*i] * cos_t - x[2*i+1] * sin_t
        x_out[2*i + 1] = x[2*i] * sin_t + x[2*i+1] * cos_t
    return x_out

n_heads = 4
scores = np.zeros((seq_len, seq_len))
for _ in range(n_heads):
    q_base = np.random.randn(d) * 0.5
    k_base = np.random.randn(d) * 0.5
    for m in range(seq_len):
        q_m = rope_rotate(q_base, m)
        for n in range(seq_len):
            k_n = rope_rotate(k_base, n)
            scores[m, n] += np.dot(q_m, k_n)
scores /= n_heads

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(scores, cmap='RdBu_r', aspect='auto')
ax.set_title('RoPE 注意力分数（对角线=相同相对距离）')
ax.set_xlabel('Key 位置 n')
ax.set_ylabel('Query 位置 m')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print("验证：对角线上（m-n 相同）的分数是否一致：")
for rel in [0, 1, 3, 7]:
    vals = [scores[i, i-rel] for i in range(rel, seq_len)]
    print(f"  m-n={rel:>2}: 标准差={np.std(vals):.8f} （应≈0）")

## 实验 3：ALiBi — 距离线性惩罚如何塑造注意力

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

seq_len = 32
slopes = [2**(-i/8) for i in range(1, 5)]

fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for idx, (ax, slope) in enumerate(zip(axes, slopes)):
    mask = np.triu(np.ones((seq_len, seq_len)), k=1) * (-1e9)
    dist = np.abs(np.arange(seq_len)[:, None] - np.arange(seq_len)[None, :])
    bias = -slope * dist + mask
    attn = softmax(bias, axis=-1)
    im = ax.imshow(attn, cmap='Blues', aspect='auto', vmin=0)
    ax.set_title(f'Head {idx+1}, slope={slope:.4f}')
    ax.set_xlabel('Key 位置')
    ax.set_ylabel('Query 位置')

plt.suptitle('ALiBi 注意力分布（slope 越大，越关注近处）', fontsize=13)
plt.tight_layout()
plt.show()

print("=== ALiBi 外推性 ===")
slope = slopes[1]
for test_len in [16, 32, 64, 128]:
    dist = np.abs(np.arange(test_len)[:, None] - np.arange(test_len)[None, :])
    mask = np.triu(np.ones((test_len, test_len)), k=1) * (-1e9)
    attn = softmax(-slope * dist + mask, axis=-1)
    last_row = attn[-1]
    top3 = np.argsort(last_row)[-3:][::-1]
    print(f"  上下文长度={test_len:>3}: 最后 token 最关注位置 {list(top3)} （始终是最近的！）")

## 实验 4：RoPE vs ALiBi — 衰减模式对比

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

def softmax(x, axis=-1):
    e = np.exp(x - x.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)

T = 64
d = 8
thetas = 10000.0 ** (-2.0 * np.arange(d // 2) / d)

def rope_rotate(x, pos):
    x_out = x.copy()
    for i in range(d // 2):
        cos_t, sin_t = np.cos(pos * thetas[i]), np.sin(pos * thetas[i])
        x_out[2*i]     = x[2*i] * cos_t - x[2*i+1] * sin_t
        x_out[2*i + 1] = x[2*i] * sin_t + x[2*i+1] * cos_t
    return x_out

np.random.seed(1)
q_base = np.random.randn(d) * 0.3
k_base = np.random.randn(d) * 0.3

rope_scores = []
for n in range(T):
    q_T = rope_rotate(q_base, T)
    k_n = rope_rotate(k_base, n)
    rope_scores.append(np.dot(q_T, k_n))

slope = 0.25
alibi_scores = np.array([-slope * (T - n) for n in range(T)])

rope_attn = softmax(np.array(rope_scores).reshape(1, -1))[0]
alibi_attn = softmax(alibi_scores.reshape(1, -1))[0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(T), rope_attn, '-', label='RoPE', alpha=0.8)
ax.plot(range(T), alibi_attn, '--', label='ALiBi (slope=0.25)', alpha=0.8)
ax.set_xlabel('Key 位置 n（query 在位置 64）')
ax.set_ylabel('注意力权重')
ax.set_title('RoPE vs ALiBi：query 对不同位置 token 的注意力分配')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("RoPE：注意力呈复杂周期性衰减（由旋转频率决定）")
print("ALiBi：注意力严格单调衰减到最近 token（线性偏置的简单效果）")